# Imports

In [28]:
import os
import json
import itables
import pandas as pd
from helpers import *

# itables.init_notebook_mode(all_interactive=True)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Wave Data Extraction
Parse raw data sets to extract questions relevant to the research topic. Relevant questions ask respondants about either military aid for Ukraine or economic sanctions on Russia.
Additionally extract fieldwork timelines during which respondants were surveyed.

In [6]:
collected = []

for file in os.listdir(EB_BASE_PATH):
    collected.append(parse_doc(file))

collected.sort(key=lambda k: k["wave_id"])
df = pd.DataFrame(collected)
df

,eb_n,wave_id,fw_start,fw_end,season,questions,q_ids
0,eb97,97.5,2022-06-17,2022-07-17,Summer 2022,"[QE2.1. The EU has taken a series of actions as a response to Russia’s invasion in Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QE2.3. The EU has taken a series of actions as a response to Russia’s invasion in Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing supply and delivery of military equipment to Ukraine]","[QE2_1, QE2_3]"
1,eb98,98.2,2023-01-12,2023-02-06,Spring 2023,"[QE2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QE2.3. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QE2_1, QE2_3]"
2,eb99,99.4,2023-05-31,2023-06-25,Spring 2023,"[QE2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QE2.3. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QE2_1, QE2_3]"
3,eb100,100.2,2023-10-23,2023-11-17,Fall 2023,"[QD2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QD2.3. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_3]"
4,eb101,101.3,2024-04-02,2024-05-09,Spring 2024,"[QD2.1. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Imposing economic sanctions on Russian government, companies and individuals, QD2.2. The EU has taken a series of actions as a response to Russia's invasion of Ukraine. To what extent you agree or disagree with each of these actions taken. :-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_2]"
5,eb102,102.2,2024-10-10,2024-11-05,Fall 2024,"[QD2.1. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken?:-Imposing economic sanctions on Russian government, companies and individuals, QD2.2. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken?:-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_2]"
6,eb103,103.3,2025-03-26,2025-04-22,Spring 2025,"[QD2.1. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken.:-Imposing economic sanctions on Russian government, companies and individuals, QD2.2. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disagree with each of these actions taken.:-Financing the purchase and supply of military equipment to Ukraine]","[QD2_1, QD2_2]"
7,eb104,104.1,2025-10-09,2025-11-05,Fall 2025,"[QD2.1. The EU has taken a series of actions as a response to Russia’s invasion of Ukraine. To what extent you agree or disa

In [17]:
if os.path.exists("data/interim/wave_questions.csv"):
    questions = pd.read_csv("data/interim/wave_questions.csv")
else:
    qs = pd.DataFrame(df["questions"].tolist(), index=df.index)
    qs.columns = ["q1", "q2"]
    qs["q1"] = qs["q1"].apply(lambda v: f"{v.split(" ", 1)[0]} {v.split(":-")[-1]}")
    qs["q2"] = qs["q2"].apply(lambda v: f"{v.split(" ", 1)[0]} {v.split(":-")[-1]}")
    questions = df.join(qs).drop(columns=["questions", "q_ids"])

    with open("data/interim/wave_questions.csv", "w") as f:
        questions.to_csv(f, index=False)

questions

,eb_n,wave_id,fw_start,fw_end,season,q1,q2
0,eb97,97.5,2022-06-17,2022-07-17,Summer 2022,"QE2.1. Imposing economic sanctions on Russian government, companies and individuals",QE2.3. Financing supply and delivery of military equipment to Ukraine
1,eb98,98.2,2023-01-12,2023-02-06,Spring 2023,"QE2.1. Imposing economic sanctions on Russian government, companies and individuals",QE2.3. Financing the purchase and supply of military equipment to Ukraine
2,eb99,99.4,2023-05-31,2023-06-25,Spring 2023,"QE2.1. Imposing economic sanctions on Russian government, companies and individuals",QE2.3. Financing the purchase and supply of military equipment to Ukraine
3,eb100,100.2,2023-10-23,2023-11-17,Fall 2023,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.3. Financing the purchase and supply of military equipment to Ukraine
4,eb101,101.3,2024-04-02,2024-05-09,Spring 2024,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
5,eb102,102.2,2024-10-10,2024-11-05,Fall 2024,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
6,eb103,103.3,2025-03-26,2025-04-22,Spring 2025,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
7,eb104,104.1,2025-10-09,2025-11-05,Fall 2025,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals",QD2.2. Financing the purchase and supply of military equipment to Ukraine
8,eb105,105.2,2026-03-12,2026-04-05,Spring 2026,"QD2.1. Imposing economic sanctions on Russian government, companies and individuals (including use of immobilised Russian assets to finance support for Ukraine)",QD2.2. Financing the purchase and supply of military equipment to Ukraine


Looking at the extracted questions from each wave, a change in the wording of the questions only happens twice: *QE2.3* in wave *97.5* and *QD2.1* in wave *105.2*. However, given that the deviations in wording are relatively minor - *97.5-QE2.3* implies the same thing as *98.2-QE2.3* and *105.2-QD2.1* clarifies the scope of the sanctions - a consistent analysis over time can still be made.

## Per-country score extraction
This takes a while to run, but saves to data/interim/wave_scores.json.

In [49]:
if os.path.exists("data/interim/wave_scores.json"):
    with open("data/interim/wave_scores.json", "r") as f:
        data = json.load(f)
else:
    data = {}

    for row in df[["eb_n", "q_ids"]].itertuples():
        data[row.eb_n] = collect_scores(row.eb_n, row.q_ids)  # type: ignore[arg-type]

    with open("data/interim/wave_scores.json", "w") as f:
        json.dump(data, f, indent=4)

series = pd.Series({
    (wave, q, COUNTRY_CODES[country]): metrics
    for wave, questions in data.items()
    for q, countries in questions.items()
    for country, metrics in countries.items()
})

wdf = pd.DataFrame(series.tolist(), index=series.index)
wdf.index.names = ['wave', 'question', 'country']

pd.set_option("display.max_rows", None)
wdf

total  total_agree  total_disagree  totally_agree  \
wave  question country                                                          
eb97  QE2_1    Belgium       1009          819             178            437   
               Bulgaria      1038          477             440            205   
               Czechia       1015          727             249            484   
               Denmark       1037          956              67            762   
               Germany       1507         1229             224            859   
               Estonia       1026          758             194            562   
               Ireland       1017          918              64            557   
               Greece        1010          603             356            278   
               Spain         1009          789             131            484   
               France        1010          700             202            362   
               Croatia       1000          776             182            399   
               Italy         1023          781             195            407   
               Cyprus         503          243             222            105   
               Latvia        1028          710             251            524   
               Lithuania     1001          796             189            482   
               Luxembourg     506          355             102            154   
               Hungary       1026          669             331            310   
               Malta          503          408              78            249   
               Netherlands   1013          889             114            589   
               Austria       1006          642             322            332   
               Poland        1016          941              61            579   
               Portugal      1009          950              26            554   
               Romania       1042          716             248            331   
               Slovenia      1001          624             341            347   
               Slovakia      1033          606             401            358   
               Finland       1045          921              82            672   
               Sweden        1035          967              63            739   
      QE2_3    Belgium       1009          732             259            320   
               Bulgaria      1038          363             598            145   
               Czechia       1015          571             419            323   
               Denmark       1037          943              82            622   
               Germany       1507         1065             369            606   
               Estonia       1026          767             191            553   
               Ireland       1017          878              89            468   
               Greece        1010          413             560            167   
               Spain         1009          671             236            369   
               France        1010          623             273            267   
               Croatia       1000          740             225            359   
               Italy         1023          582             378            237   
               Cyprus         503          222             248             87   
               Latvia        1028          739             224            512   
               Lithuania     1001          827             161            491   
               Luxembourg     506          315             152            126   
               Hungary       1026          590             411            240   
               Malta          503          369             117            229   
               Netherlands   1013          855             143            527   
               Austria       1006          506             460            227   
               Poland        1016          927             